# Data Overview And Transformations

This notebook compares the raw merged panel with the transformed research panel used in the paper. It is meant for checking sample coverage, variable construction, missingness, stationarity, and the event windows that motivate the January 2022 regime split.


In [ ]:
from __future__ import annotations

from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "pyproject.toml").exists():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise RuntimeError("Could not locate the project root containing pyproject.toml.")
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_PATH = PROJECT_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Image, display

pd.options.display.max_columns = 50
pd.options.display.float_format = lambda value: f"{value:,.3f}"
plt.style.use("seaborn-v0_8-whitegrid")

from laos_fx_oil_macro.data import (
    DISTRESS_BREAK,
    ProjectPaths,
    enrich_panel_with_alt_gpr,
    event_snapshot_table,
    load_panel,
    summarize_stationarity,
    transform_panel,
)
from laos_fx_oil_macro.notebook_helpers import (
    build_notebook_project,
    load_csv,
    plot_transformed_series,
    round_numeric_columns,
    variable_definitions_table,
)

project = build_notebook_project(PROJECT_ROOT)
EVENT_DATES = ["2022-03-31", "2023-10-31", "2026-02-28"]
PLOT_COLUMNS = ["GPR_ME", "oil_shock", "fx_dep", "inflation_mom", "CPI_Inf"]
project.root


In [ ]:
paths = ProjectPaths(PROJECT_ROOT)
raw_panel = load_panel(paths.panel_path)
gpr_components = load_panel(paths.gpr_components_path)
rebuilt_panel = enrich_panel_with_alt_gpr(transform_panel(raw_panel), gpr_components)
processed_panel = load_panel(project.processed_panel) if project.processed_panel.exists() else rebuilt_panel.copy()

overview = pd.DataFrame(
    [
        {
            "panel": "raw merged panel",
            "rows": len(raw_panel),
            "columns": raw_panel.shape[1],
            "start": raw_panel["Date"].min().date(),
            "end": raw_panel["Date"].max().date(),
        },
        {
            "panel": "processed research panel",
            "rows": len(processed_panel),
            "columns": processed_panel.shape[1],
            "start": processed_panel["Date"].min().date(),
            "end": processed_panel["Date"].max().date(),
        },
    ]
)
display(overview)
print("Raw panel preview")
display(raw_panel.head(5))
print("Processed panel preview")
display(processed_panel.head(5))


In [ ]:
definitions = variable_definitions_table()
display(definitions)


In [ ]:
sample_window = pd.DataFrame(
    {
        "metric": ["observations", "raw start", "raw end", "processed start", "processed end", "regime break"],
        "value": [
            len(processed_panel),
            raw_panel["Date"].min().date(),
            raw_panel["Date"].max().date(),
            processed_panel["Date"].min().date(),
            processed_panel["Date"].max().date(),
            DISTRESS_BREAK.date(),
        ],
    }
)
missingness = (
    processed_panel.isna()
    .sum()
    .rename("missing_obs")
    .to_frame()
    .assign(share=lambda frame: frame["missing_obs"] / len(processed_panel))
    .query("missing_obs > 0")
    .reset_index(names="variable")
)
summary_stats = processed_panel[["GPR_ME", "oil_shock", "fx_dep", "inflation_mom", "CPI_Inf"]].agg(
    ["count", "mean", "std", "min", "median", "max"]
).T

display(sample_window)
print("Missingness in the processed panel")
display(round_numeric_columns(missingness))
print("Summary statistics for the main paper variables")
display(round_numeric_columns(summary_stats.reset_index().rename(columns={"index": "variable"})))


In [ ]:
stationarity_path = project.tables_dir / "stationarity.csv"
if stationarity_path.exists():
    stationarity = load_csv(stationarity_path)
else:
    stationarity = summarize_stationarity(
        rebuilt_panel,
        ["USDLAK", "CPI", "CPI_Inf", "GPR_ME", "GPR_ME_alt", "oil_shock", "fx_dep", "inflation_mom"],
    )
display(round_numeric_columns(stationarity))


## Why the models do not use `USDLAK` and `CPI` levels directly

The dynamic models are run on transformed series because the level variables behave like trending macro time series rather than stable fluctuations around a mean. In the stationarity table, `USDLAK` and `CPI` are clearly non-stationary in levels, so using them directly in local projections or the TVP-SV-VAR would blur the shock-transmission interpretation with trend behavior. The paper therefore works with `fx_dep = 100*Δlog(USDLAK)` and `inflation_mom = 100*Δlog(CPI)` for the benchmark dynamic specifications, while `CPI_Inf` is retained for crisis context and robustness checks.


In [ ]:
fig = plot_transformed_series(processed_panel, columns=PLOT_COLUMNS, event_dates=EVENT_DATES)
display(fig)
plt.close(fig)


## Regime split and event windows

The vertical break in January 2022 marks the debt-distress and exchange-rate stress regime used throughout the paper. The event dates below anchor the descriptive windows and the scenario IRFs used later in the TVP-SV-VAR notebook.


In [ ]:
event_dates = pd.DataFrame({"event_date": EVENT_DATES})
display(event_dates)

event_windows = event_snapshot_table(processed_panel, EVENT_DATES, window=2)
display(round_numeric_columns(event_windows))
